# Coverage Reconciliation

Baseline GPS evidence indicates physical presence; e-tally indicates reported service delivery. The reconciled product is the operational decision source.

In [1]:
from pathlib import Path
import sys, pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from src.ingestion.db_connection import get_connection
from src.project_paths import output_table_path

In [2]:
with get_connection() as connection:
    settlement = pd.read_sql_query('SELECT * FROM processed.settlement_coverage_reconciliation ORDER BY settlement_id', connection)
    ward = pd.read_sql_query('SELECT * FROM processed.ward_coverage_reconciliation ORDER BY ward_code', connection)
    unmatched = pd.read_sql_query('SELECT * FROM raw.etally_records_unmatched ORDER BY source_row_number', connection)

# All summary figures below are computed directly from `settlement`, not hardcoded --
# a prior version of this cell hardcoded these values as literals, which meant they
# would silently go stale if the underlying reconciliation table changed (as it did
# once the source_file_date_mismatch GPS defect fix was applied). See
# technical_decisions.md for that finding.
planned_settlements = len(settlement)
gps_visited = int((settlement.gps_visit_status == 'visited').sum())
etally_reported = int((settlement.etally_report_status == 'reported').sum())
ambiguous_gps_cases = int((settlement.gps_visit_status == 'ambiguous').sum())
disagreement = int(settlement.discrepancy_flag.sum())
strict_agreement = int(((settlement.gps_visit_status != 'ambiguous') & (~settlement.discrepancy_flag)).sum())
definitive_total = strict_agreement + disagreement
definitive_agreement_rate_pct = round(100 * strict_agreement / definitive_total, 2) if definitive_total else float('nan')
gps_coverage_pct = round(100 * gps_visited / planned_settlements, 2)
etally_coverage_pct = round(100 * etally_reported / planned_settlements, 2)
absolute_difference_pp = round(abs(gps_coverage_pct - etally_coverage_pct), 2)
dose_over_target = settlement.loc[settlement.dose_quality_flag]
linked_dose_over_target_rows = int(len(dose_over_target))
linked_dose_over_target_doses = int((dose_over_target.reported_doses_all_linked - dose_over_target.reported_doses_plausible_only).sum())

summary = pd.DataFrame([{
    'planned_settlements': planned_settlements,
    'gps_visited': gps_visited,
    'etally_reported': etally_reported,
    'strict_agreement': strict_agreement,
    'disagreement': disagreement,
    'ambiguous_gps_cases': ambiguous_gps_cases,
    'definitive_agreement_rate_pct': definitive_agreement_rate_pct,
    'gps_coverage_pct': gps_coverage_pct,
    'etally_coverage_pct': etally_coverage_pct,
    'absolute_difference_pp': absolute_difference_pp,
    'linked_dose_over_target_rows': linked_dose_over_target_rows,
    'linked_dose_over_target_doses': linked_dose_over_target_doses,
}])
settlement.to_csv(output_table_path('settlement_coverage_reconciliation.csv'), index=False)
ward.to_csv(output_table_path('ward_coverage_reconciliation.csv'), index=False)
summary.to_csv(output_table_path('reconciliation_summary.csv'), index=False)
unmatched.to_csv(output_table_path('unmatched_etally_records.csv'), index=False)
summary

   planned_settlements  ...  linked_dose_over_target_doses
0                 2562  ...                          27551

[1 rows x 12 columns]